# __Chapter 5. Roots: Bracketing Methods__

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


### [Algorithm] 증분 탐색법

- 증분 탐색법 함수 정의

In [ ]:
import numpy as np

def find_all_brackets(func, a, b, n):
    """
    주어진 구간 [a, b]를 n개의 동일한 구간으로 나누고,
    f(x)의 부호 변화가 발생하는 모든 구간을 찾는다.

    Parameters
    ----------
    func : function
        조사할 함수 f(x)
    a : float
        시작점
    b : float
        끝점
    n : int
        등분할 구간의 개수

    Returns
    -------
    brackets : list of tuples
        근이 존재할 가능성이 있는 구간 [(x1, x2), ...]
    """

    # [a, b]를 n개의 동일한 구간으로 분할
    xs = np.linspace(a, b, n)

    # 함수값 계산
    fs = func(xs)

    brackets = []

    # 각 소구간에서 부호 변화 검사
    for i in range(n - 1):

        # 왼쪽 끝점이 정확한 근인 경우
        if fs[i] == 0:
            brackets.append((xs[i], xs[i]))

        # 두 끝점 사이에서 부호가 바뀌는 경우
        elif fs[i] * fs[i + 1] < 0:
            brackets.append((xs[i], xs[i + 1]))

    # 마지막 점이 정확한 근인 경우
    if fs[-1] == 0:
        brackets.append((xs[-1], xs[-1]))

    return brackets

- 예제 5.2 - 증분 탐색법

$$ f(x) = sin(10x) + cos(3x) $$

In [ ]:
import numpy as np

def f(x):
    return np.sin(10*x) + np.cos(3*x)

# 탐색 구간과 분할 개수
a = 3
b = 6
n = 50

# 증분 탐색
brackets = find_all_brackets(f, a, b, n)

# 결과 출력
print("number of brackets:", len(brackets))
for a, b in brackets:
    print(f"[{a:.4f}, {b:.4f}]")

### [Algorithm] 이분법

- 이분법 함수 정의

In [ ]:
import pandas as pd

def bisection(func, a, b, func_tol=1e-10, ea_tol=1e-6,
              max_iter=200, track=False):
    """
    브래킷 [a, b]에서 이분법을 이용해 func의 영점을 찾는다.

    Parameters
    ----------
    func : function
        f(x)
    a, b : float
        초기 브래킷
    func_tol : float
        |f(c)|에 대한 허용오차
    ea_tol : float
        근사 백분율 상대오차의 허용값 (%)
    max_iter : int
        최대 반복 횟수
    track : bool
        True이면 반복 이력을 DataFrame으로 반환

    Returns
    -------
    root : float
        근의 근사값
    f_root : float
        근에서의 함수값
    iterations : int
        반복 횟수
    history : pandas.DataFrame or None
        반복 이력
    """

    fa = func(a)
    fb = func(b)

    # 끝점 자체가 근인 경우
    if fa == 0:
        return a, fa, 0, pd.DataFrame() if track else None

    if fb == 0:
        return b, fb, 0, pd.DataFrame() if track else None

    # 유효한 bracket인지 확인
    if fa * fb > 0:
        raise ValueError(
            "초기 구간이 영점을 끼고 있지 않습니다 "
            "(f(a) * f(b) > 0)."
        )

    hist = []
    c_old = None

    for k in range(1, max_iter + 1):

        # 중점
        c = (a + b) / 2.0
        fc = func(c)

        # 근사 백분율 상대오차
        if c_old is None or c == 0:
            ea = None
        else:
            ea = abs((c - c_old) / c) * 100

        # 반복 이력 저장
        if track:
            hist.append({
                "iter": k,
                "a": a,
                "b": b,
                "c": c,
                "f(a)": fa,
                "f(b)": fb,
                "f(c)": fc,
                "ea(%)": ea
            })

        # 정확한 근
        if fc == 0:
            return (
                c, fc, k,
                pd.DataFrame(hist) if track else None
            )

        # 수렴 판정
        if abs(fc) < func_tol:
            return (
                c, fc, k,
                pd.DataFrame(hist) if track else None
            )

        if ea is not None and ea < ea_tol:
            return (
                c, fc, k,
                pd.DataFrame(hist) if track else None
            )

        # 브래킷 갱신
        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

        c_old = c

    return (
        c, fc, max_iter,
        pd.DataFrame(hist) if track else None
    )

- 예제 5.3 - 자유낙하 4초 후 속도가 36 m/s가 되는 사람의 질량 구하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
g = 9.81       # gravitational acceleration (m/s^2)
cd = 0.25      # drag coefficient (kg/m)
t = 4.0        # time (s)
v_target = 36  # target velocity (m/s)

# Function
def f(m):
    return np.sqrt(g * m / cd) * np.tanh(
        np.sqrt(g * cd / m) * t
    ) - v_target

# Mass range
m = np.linspace(50, 200, 1000)

# Plot
plt.figure(figsize=(6, 4))

plt.plot(m, f(m), linewidth=2)
plt.axhline(0, color='black', linestyle='--', linewidth=1)

plt.xlabel('Mass, m (kg)')
plt.ylabel('f(m)')
plt.title('Root of the Bungee Jumper Problem')

plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq


# ============================================================
# 1. Parameters
# ============================================================

g = 9.81          # gravitational acceleration (m/s^2)
cd = 0.25         # drag coefficient (kg/m)
t = 4.0           # time (s)
v_target = 36.0   # target velocity (m/s)


# ============================================================
# 2. Function
# ============================================================

def f(m):
    return (
        np.sqrt(g * m / cd)
        * np.tanh(np.sqrt(g * cd / m) * t)
        - v_target
    )


# ============================================================
# 3. True root
#    Used only for calculating the true percent relative error
# ============================================================

x_true = brentq(f, 50, 200)

print(f"True root = {x_true:.4f} kg")


# ============================================================
# 4. Bisection method
#
#    앞의 [Algorithm] 이분법에서 정의한 bisection() 함수를 사용한다.
# ============================================================

xl = 50.0
xu = 200.0

es = 0.5      # stopping criterion (%)

xr, fxr, iteration, hist_df = bisection(
    f, xl, xu,
    func_tol=0.0,   # 함수값 기준으로는 종료하지 않고, 근사 상대오차(es)만으로 종료
    ea_tol=es,
    track=True
)

# bisection()이 반환한 이력(a, b, c, ea(%))을 원래 출력 형식(x_l, x_u, x_r, ea (%))으로 변환
history = hist_df.rename(columns={
    "iter": "iteration",
    "a": "x_l",
    "b": "x_u",
    "c": "x_r",
    "ea(%)": "ea (%)"
})[["iteration", "x_l", "x_u", "x_r", "ea (%)"]]

# True percent relative error 추가
history["et (%)"] = (history["x_r"] - x_true).abs() / abs(x_true) * 100

ea = history["ea (%)"].iloc[-1]
et = history["et (%)"].iloc[-1]


# ============================================================
# 5. Print results
# ============================================================

result = pd.DataFrame(history)

print()
print(result.to_string(
    index=False,
    formatters={
        "x_l": "{:.4f}".format,
        "x_u": "{:.4f}".format,
        "x_r": "{:.4f}".format,
        "ea (%)": lambda x: "-" if np.isnan(x) else f"{x:.2f}",
        "et (%)": "{:.2f}".format
    }
))

print()
print(f"Estimated mass = {xr:.4f} kg")
print(f"Approximate percent relative error = {ea:.2f} %")
print(f"True percent relative error = {et:.2f} %")


# ============================================================
# 6. Error comparison plots
# ============================================================

# Font sizes
FS_TITLE = 20
FS_LABEL = 18
FS_TICK = 18
FS_LEGEND = 16

iterations = result["iteration"]
ea_values = result["ea (%)"]
et_values = result["et (%)"]

fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 6)
)


# ------------------------------------------------------------
# (a) Linear scale
# ------------------------------------------------------------

axes[0].plot(
    iterations,
    ea_values,
    marker="o",
    linewidth=2,
    label="Approximate error"
)

axes[0].plot(
    iterations,
    et_values,
    marker="s",
    linewidth=2,
    label="True error"
)

axes[0].axhline(
    es,
    linestyle="--",
    linewidth=1.5,
    label="Stopping criterion (0.5%)"
)

axes[0].set_xlabel("Iteration", fontsize=FS_LABEL)
axes[0].set_ylabel("Percent Relative Error (%)", fontsize=FS_LABEL)
axes[0].set_title("Linear Scale", fontsize=FS_TITLE)

axes[0].tick_params(axis="both", labelsize=FS_TICK)

axes[0].set_xticks(iterations)
axes[0].grid(True)
axes[0].legend(fontsize=FS_LEGEND)


# ------------------------------------------------------------
# (b) Logarithmic scale
# ------------------------------------------------------------

axes[1].plot(
    iterations,
    ea_values,
    marker="o",
    linewidth=2,
    label="Approximate error")

axes[1].plot(
    iterations,
    et_values,
    marker="s",
    linewidth=2,
    label="True error"
)

axes[1].axhline(
    es,
    linestyle="--",
    linewidth=1.5,
    label="Stopping criterion (0.5%)"
)

axes[1].set_yscale("log")

axes[1].set_xlabel("Iteration", fontsize=FS_LABEL)
axes[1].set_ylabel("Percent Relative Error (%)", fontsize=FS_LABEL)
axes[1].set_title("Logarithmic Scale", fontsize=FS_TITLE)

axes[1].tick_params(axis="both", labelsize=FS_TICK)

axes[1].set_xticks(iterations)
axes[1].grid(True, which="both")
axes[1].legend(fontsize=FS_LEGEND)

plt.tight_layout()
plt.show()

### [Algorithm] 가위치법

- 가위치법 함수 정의

In [ ]:
import pandas as pd

def false_position(func, a, b, func_tol=1e-10, ea_tol=1e-6,
                   max_iter=200, track=False):
    """
    브래킷 [a, b]에서 가위치법(false-position method)을 이용해
    func의 영점을 찾는다.
    """

    fa = func(a)
    fb = func(b)

    # 끝점 자체가 근인 경우
    if fa == 0:
        return a, fa, 0, pd.DataFrame() if track else None

    if fb == 0:
        return b, fb, 0, pd.DataFrame() if track else None

    # 유효한 bracket인지 확인
    if fa * fb > 0:
        raise ValueError(
            "초기 구간이 영점을 끼고 있지 않습니다 "
            "(f(a) * f(b) > 0)."
        )

    hist = []
    c_old = None

    for k in range(1, max_iter + 1):

        # 분모 확인
        if fb == fa:
            raise ZeroDivisionError(
                "f(a)와 f(b)가 같아 가위치법을 적용할 수 없습니다."
            )

        # 선형보간에 의한 x축 교점
        c = b - fb * (b - a) / (fb - fa)
        fc = func(c)

        # 근사 백분율 상대오차
        if c_old is None or c == 0:
            ea = None
        else:
            ea = abs((c - c_old) / c) * 100

        # 반복 이력 저장
        if track:
            hist.append({
                "iter": k,
                "a": a,
                "b": b,
                "c": c,
                "f(a)": fa,
                "f(b)": fb,
                "f(c)": fc,
                "ea(%)": ea
            })

        # 정확한 근
        if fc == 0:
            return (
                c, fc, k,
                pd.DataFrame(hist) if track else None
            )

        # 수렴 판정
        if abs(fc) < func_tol:
            return (
                c, fc, k,
                pd.DataFrame(hist) if track else None
            )

        if ea is not None and ea < ea_tol:
            return (
                c, fc, k,
                pd.DataFrame(hist) if track else None
            )

        # 브래킷 갱신
        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

        c_old = c

    return (
        c, fc, max_iter,
        pd.DataFrame(hist) if track else None
    )

- 예제 5.5 - 가위치법으로 동일 문제 풀기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq


# ============================================================
# 1. Parameters
# ============================================================

g = 9.81          # gravitational acceleration (m/s^2)
cd = 0.25         # drag coefficient (kg/m)
t = 4.0           # time (s)
v_target = 36.0   # target velocity (m/s)

es = 0.5          # stopping criterion (%)


# ============================================================
# 2. Function
# ============================================================

def f(m):
    return (
        np.sqrt(g * m / cd)
        * np.tanh(np.sqrt(g * cd / m) * t)
        - v_target
    )


# ============================================================
# 3. True root
#    Used only for calculating the true percent relative error
# ============================================================

x_true = brentq(f, 50, 200)

print(f"True root = {x_true:.4f} kg")


# ============================================================
# 4. False-position method
#
#    앞의 [Algorithm] 가위치법에서 정의한 false_position() 함수를 사용한다.
# ============================================================

xl = 50.0
xu = 200.0

xr, fxr, iteration, hist_df = false_position(
    f, xl, xu,
    func_tol=0.0,   # 함수값 기준으로는 종료하지 않고, 근사 상대오차(es)만으로 종료
    ea_tol=es,
    track=True
)

# false_position()이 반환한 이력(a, b, c, ea(%))을 원래 출력 형식(x_l, x_u, x_r, ea (%))으로 변환
history = hist_df.rename(columns={
    "iter": "iteration",
    "a": "x_l",
    "b": "x_u",
    "c": "x_r",
    "ea(%)": "ea (%)"
})[["iteration", "x_l", "x_u", "x_r", "ea (%)"]]

# True percent relative error 추가
history["et (%)"] = (history["x_r"] - x_true).abs() / abs(x_true) * 100

ea = history["ea (%)"].iloc[-1]
et = history["et (%)"].iloc[-1]


# ============================================================
# 5. Print results
# ============================================================

result = pd.DataFrame(history)

print()
print(result.to_string(
    index=False,
    formatters={
        "x_l": "{:.4f}".format,
        "x_u": "{:.4f}".format,
        "x_r": "{:.4f}".format,
        "ea (%)": lambda x: "-" if np.isnan(x) else f"{x:.2f}",
        "et (%)": "{:.2f}".format
    }
))

print()
print(f"Estimated mass = {xr:.4f} kg")
print(f"Approximate percent relative error = {ea:.2f} %")
print(f"True percent relative error = {et:.2f} %")


# ============================================================
# 6. Error comparison plots
# ============================================================

iterations = result["iteration"]
ea_values = result["ea (%)"]
et_values = result["et (%)"]


# Font sizes
FS_TITLE = 20
FS_LABEL = 18
FS_TICK = 16
FS_LEGEND = 14


fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 6)
)


# ------------------------------------------------------------
# (a) Linear scale
# ------------------------------------------------------------

axes[0].plot(
    iterations,
    ea_values,
    marker="o",
    linewidth=2,
    label="Approximate percent relative error"
)

axes[0].plot(
    iterations,
    et_values,
    marker="s",
    linewidth=2,
    label="True percent relative error"
)

axes[0].axhline(
    es,
    linestyle="--",
    linewidth=1.5,
    label="Stopping criterion (0.5%)"
)

axes[0].set_xlabel("Iteration", fontsize=FS_LABEL)
axes[0].set_ylabel(
    "Percent Relative Error (%)",
    fontsize=FS_LABEL
)
axes[0].set_title(
    "Linear Scale",
    fontsize=FS_TITLE
)

axes[0].set_xticks(iterations)
axes[0].tick_params(
    axis="both",
    labelsize=FS_TICK
)

axes[0].grid(True)
axes[0].legend(fontsize=FS_LEGEND)


# ------------------------------------------------------------
# (b) Logarithmic scale
# ------------------------------------------------------------

axes[1].plot(
    iterations,
    ea_values,
    marker="o",
    linewidth=2,
    label="Approximate percent relative error"
)

axes[1].plot(
    iterations,
    et_values,
    marker="s",
    linewidth=2,
    label="True percent relative error"
)

axes[1].axhline(
    es,
    linestyle="--",
    linewidth=1.5,
    label="Stopping criterion (0.5%)"
)

axes[1].set_yscale("log")

axes[1].set_xlabel(
    "Iteration",
    fontsize=FS_LABEL
)
axes[1].set_ylabel(
    "Percent Relative Error (%)",
    fontsize=FS_LABEL
)
axes[1].set_title(
    "Logarithmic Scale",
    fontsize=FS_TITLE
)

axes[1].set_xticks(iterations)
axes[1].tick_params(
    axis="both",
    labelsize=FS_TICK
)

axes[1].grid(
    True,
    which="both"
)
axes[1].legend(
    fontsize=FS_LEGEND
)


plt.tight_layout()
plt.show()

- 예제 5.6 - 가위치법이 안좋은 경우

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. Function
# ============================================================

def f(x):
    return x**10 - 1


# ============================================================
# 2. Initial bracket
# ============================================================

xl = 0.0
xu = 1.3

es = 0.5       # stopping criterion (%)
max_iter = 5


# ============================================================
# 3. False-position method
#
#    앞의 [Algorithm] 가위치법에서 정의한 false_position() 함수를 사용한다.
#    max_iter=5로 두어, 이 예제가 보여주려는 "느린/나쁜 수렴"을
#    처음 5번 반복까지만 확인한다.
# ============================================================

xr_fp, fxr_fp, iter_fp, hist_fp_raw = false_position(
    f, xl, xu,
    func_tol=0.0,   # 함수값 기준으로는 종료하지 않음
    ea_tol=es,
    max_iter=max_iter,
    track=True
)

history_fp = hist_fp_raw.rename(columns={
    "iter": "iteration",
    "a": "x_l",
    "b": "x_u",
    "c": "x_r",
    "ea(%)": "ea (%)"
})[["iteration", "x_l", "x_u", "x_r", "ea (%)"]]

# True percent relative error 추가 (True root = 1)
history_fp["et (%)"] = (history_fp["x_r"] - 1.0).abs() / 1.0 * 100

ea_fp = history_fp["ea (%)"].iloc[-1]
et_fp = history_fp["et (%)"].iloc[-1]


# ============================================================
# 4. Bisection method
#
#    앞의 [Algorithm] 이분법에서 정의한 bisection() 함수를 사용해
#    같은 구간·같은 반복 횟수로 비교한다.
# ============================================================

xr_bis, fxr_bis, iter_bis, hist_bis_raw = bisection(
    f, xl, xu,
    func_tol=0.0,
    ea_tol=es,
    max_iter=max_iter,
    track=True
)

history_bis = hist_bis_raw.rename(columns={
    "iter": "iteration",
    "a": "x_l",
    "b": "x_u",
    "c": "x_r",
    "ea(%)": "ea (%)"
})[["iteration", "x_l", "x_u", "x_r", "ea (%)"]]

history_bis["et (%)"] = (history_bis["x_r"] - 1.0).abs() / 1.0 * 100

ea_bis = history_bis["ea (%)"].iloc[-1]
et_bis = history_bis["et (%)"].iloc[-1]


# ============================================================
# 5. Print results
# ============================================================

formatters = {
    "x_l": "{:.4f}".format,
    "x_u": "{:.4f}".format,
    "x_r": "{:.4f}".format,
    "ea (%)": lambda x: "-" if np.isnan(x) else f"{x:.2f}",
    "et (%)": "{:.2f}".format
}

print("=== False-position method ===")
print(history_fp.to_string(index=False, formatters=formatters))
print()
print(f"Estimated root = {xr_fp:.4f}")
print(f"Approximate percent relative error = {ea_fp:.2f} %")
print(f"True percent relative error = {et_fp:.2f} %")

print()
print("=== Bisection method ===")
print(history_bis.to_string(index=False, formatters=formatters))
print()
print(f"Estimated root = {xr_bis:.4f}")
print(f"Approximate percent relative error = {ea_bis:.2f} %")
print(f"True percent relative error = {et_bis:.2f} %")
